# Task 4: Machine Translation (Multi30k EN→DE)
**CENG 467 — NLU&G Take-Home Midterm | Student ID: 300201071**

Models: Seq2Seq + Bahdanau Attention, Helsinki-NLP Transformer

Metrics: BLEU, METEOR, ChrF, BERTScore

In [1]:
!pip install -q "datasets<3.0.0" datasets transformers sacrebleu bert-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset
import sacrebleu, nltk
nltk.download('wordnet',quiet=True); nltk.download('omw-1.4',quiet=True)
nltk.download('punkt',quiet=True); nltk.download('punkt_tab',quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from transformers import MarianMTModel, MarianTokenizer

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## 1. Load Multi30k

In [3]:
ds = load_dataset('bentrevett/multi30k')
print(f'Train: {len(ds["train"])}, Val: {len(ds["validation"])}, Test: {len(ds["test"])}')

def build_vocab(split, key, min_freq=2):
    c = Counter()
    for ex in split: c.update(ex[key].lower().strip().split())
    v = {'<pad>':0,'<sos>':1,'<eos>':2,'<unk>':3}
    for w,cnt in c.most_common():
        if cnt>=min_freq: v[w]=len(v)
    print(f'  {key} vocab: {len(v)}'); return v

src_v = build_vocab(ds['train'],'en')
tgt_v = build_vocab(ds['train'],'de')

ML=40
def enc_sent(s,v,ml):
    ids = [v['<sos>']]+[v.get(w,v['<unk>']) for w in s.lower().strip().split()]+[v['<eos>']]
    ids=ids[:ml]; l=len(ids); ids+=[0]*(ml-l); return ids,l

def prep(split):
    S,T,SL,TL=[],[],[],[]
    for ex in split:
        si,sl=enc_sent(ex['en'],src_v,ML); ti,tl=enc_sent(ex['de'],tgt_v,ML)
        S.append(si);T.append(ti);SL.append(sl);TL.append(tl)
    return np.array(S,dtype=np.int64),np.array(T,dtype=np.int64),np.array(SL,dtype=np.int64),np.array(TL,dtype=np.int64)

Str,Ttr,SLtr,TLtr=prep(ds['train'])
Sv,Tv,SLv,TLv=prep(ds['validation'])
St,Tt,SLt,TLt=prep(ds['test'])

class TDS(Dataset):
    def __init__(s,a,b,c,d): s.a,s.b,s.c,s.d=torch.LongTensor(a),torch.LongTensor(b),torch.LongTensor(c),torch.LongTensor(d)
    def __len__(s): return len(s.a)
    def __getitem__(s,i): return s.a[i],s.b[i],s.c[i],s.d[i]

BS=64
tr_dl=DataLoader(TDS(Str,Ttr,SLtr,TLtr),batch_size=BS,shuffle=True)
vl_dl=DataLoader(TDS(Sv,Tv,SLv,TLv),batch_size=BS)
ts_dl=DataLoader(TDS(St,Tt,SLt,TLt),batch_size=BS)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 29000, Val: 1014, Test: 1000
  en vocab: 7704
  de vocab: 9597


## 2. Seq2Seq + Bahdanau Attention

In [4]:
class Encoder(nn.Module):
    def __init__(s,vs,ed=256,hd=512,do=0.5):
        super().__init__()
        s.emb=nn.Embedding(vs,ed); s.rnn=nn.GRU(ed,hd,bidirectional=True,batch_first=True)
        s.fc=nn.Linear(hd*2,hd); s.drop=nn.Dropout(do)
    def forward(s,x):
        o,h=s.rnn(s.drop(s.emb(x))); h=torch.tanh(s.fc(torch.cat((h[-2],h[-1]),dim=1))); return o,h

class Attention(nn.Module):
    def __init__(s,hd=512):
        super().__init__()
        s.We=nn.Linear(hd*2,hd); s.Wd=nn.Linear(hd,hd); s.v=nn.Linear(hd,1,bias=False)
    def forward(s,dh,eo):
        e=torch.tanh(s.Wd(dh).unsqueeze(1)+s.We(eo)); return F.softmax(s.v(e).squeeze(2),dim=1)

class Decoder(nn.Module):
    def __init__(s,vs,ed=256,hd=512,do=0.5):
        super().__init__()
        s.emb=nn.Embedding(vs,ed); s.attn=Attention(hd)
        s.rnn=nn.GRU(ed+hd*2,hd,batch_first=True)
        s.fc=nn.Linear(hd+hd*2+ed,vs); s.drop=nn.Dropout(do)
    def forward(s,inp,h,eo):
        e=s.drop(s.emb(inp)); a=s.attn(h,eo); c=torch.bmm(a.unsqueeze(1),eo)
        o,h=s.rnn(torch.cat((e,c),dim=2),h.unsqueeze(0)); h=h.squeeze(0)
        p=s.fc(torch.cat((o.squeeze(1),c.squeeze(1),e.squeeze(1)),dim=1)); return p,h,a

class Seq2Seq(nn.Module):
    def __init__(s,enc,dec,dev):
        super().__init__(); s.enc=enc; s.dec=dec; s.dev=dev
    def forward(s,src,tgt,tf=0.5):
        B=src.size(0); TL=tgt.size(1); VS=s.dec.fc.out_features
        out=torch.zeros(B,TL,VS).to(s.dev); eo,h=s.enc(src); inp=tgt[:,0].unsqueeze(1)
        for t in range(1,TL):
            p,h,_=s.dec(inp,h,eo); out[:,t]=p
            inp=tgt[:,t].unsqueeze(1) if random.random()<tf else p.argmax(1).unsqueeze(1)
        return out

enc=Encoder(len(src_v)); dec=Decoder(len(tgt_v))
s2s=Seq2Seq(enc,dec,DEVICE).to(DEVICE)
opt=torch.optim.Adam(s2s.parameters(),lr=1e-3)
crit=nn.CrossEntropyLoss(ignore_index=0)
print(f'Seq2Seq params: {sum(p.numel() for p in s2s.parameters()):,}')

for ep in range(8):
    s2s.train(); tl=0
    for s,t,sl,tl_ in tqdm(tr_dl, desc=f'Ep {ep+1}/8', leave=False):
        s,t=s.to(DEVICE),t.to(DEVICE)
        opt.zero_grad(); o=s2s(s,t)
        loss=crit(o[:,1:].reshape(-1,o.size(-1)),t[:,1:].reshape(-1))
        loss.backward(); nn.utils.clip_grad_norm_(s2s.parameters(),1.0); opt.step(); tl+=loss.item()
    s2s.eval(); vl=0
    with torch.no_grad():
        for s,t,sl,tl_ in vl_dl:
            s,t=s.to(DEVICE),t.to(DEVICE); o=s2s(s,t,tf=0)
            vl+=crit(o[:,1:].reshape(-1,o.size(-1)),t[:,1:].reshape(-1)).item()
    print(f'  Ep {ep+1}: Train={tl/len(tr_dl):.4f} Val={vl/len(vl_dl):.4f}')

Seq2Seq params: 28,070,269


Ep 1/8:   0%|          | 0/454 [00:00<?, ?it/s]

  Ep 1: Train=4.2567 Val=3.4900


Ep 2/8:   0%|          | 0/454 [00:00<?, ?it/s]

  Ep 2: Train=3.0099 Val=3.3894


...

  Ep 6: Train=1.7922 Val=3.4628


Ep 7/8:   0%|          | 0/454 [00:00<?, ?it/s]

  Ep 7: Train=1.6883 Val=3.5716


Ep 8/8:   0%|          | 0/454 [00:00<?, ?it/s]

  Ep 8: Train=1.6214 Val=3.5461


In [5]:
idx2w = {v:k for k,v in tgt_v.items()}
s2s.eval(); s2s_trans=[]
test_en = ds['test']['en']; test_de = ds['test']['de']

for sent in tqdm(test_en, desc='Seq2Seq Translate'):
    ids = [src_v['<sos>']]+[src_v.get(w,src_v['<unk>']) for w in sent.lower().strip().split()]+[src_v['<eos>']]
    ids = ids[:ML]; ids += [0]*(ML-len(ids))
    src_t = torch.LongTensor([ids]).to(DEVICE)
    with torch.no_grad():
        eo,h = s2s.enc(src_t); inp=torch.LongTensor([[tgt_v['<sos>']]]).to(DEVICE); words=[]
        for _ in range(50):
            p,h,_ = s2s.dec(inp,h,eo); top=p.argmax(1).item()
            if top==tgt_v['<eos>']: break
            words.append(idx2w.get(top,'<unk>')); inp=torch.LongTensor([[top]]).to(DEVICE)
    s2s_trans.append(' '.join(words))
print(f'Translated {len(s2s_trans)} sentences')

Seq2Seq Translate:   0%|          | 0/1000 [00:00<?, ?it/s]

Translated 1000 sentences


## 3. Helsinki-NLP Transformer

In [6]:
mt_tok = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-de')
mt_model = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-de').to(DEVICE)
mt_model.eval()

hel_trans = []
for sent in tqdm(test_en, desc='Helsinki-NLP'):
    inp = mt_tok(sent, return_tensors='pt', max_length=128, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = mt_model.generate(**inp, max_length=128, num_beams=4)
    hel_trans.append(mt_tok.decode(out[0], skip_special_tokens=True))
print(f'Translated {len(hel_trans)} sentences')

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Helsinki-NLP:   0%|          | 0/1000 [00:00<?, ?it/s]

Translated 1000 sentences


## 4. Evaluation

In [7]:
def eval_mt(preds, refs, name):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    mets = [nltk_meteor([word_tokenize(r.lower())], word_tokenize(p.lower())) for p,r in zip(preds,refs)]
    meteor = np.mean(mets)
    from bert_score import score as bscore
    _,_,F1 = bscore(preds, refs, lang='de', verbose=False)
    bs = F1.mean().item()
    res = {'bleu':bleu,'chrf':chrf,'meteor':meteor,'bertscore':bs}
    print(f'\n{name}:')
    for k,v in res.items(): print(f'  {k}: {v:.4f}')
    return res

s2s_res = eval_mt(s2s_trans, test_de, 'Seq2Seq+Attention')
hel_res = eval_mt(hel_trans, test_de, 'Helsinki-NLP')

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seq2Seq+Attention:
  bleu: 3.7530
  chrf: 39.8409
  meteor: 0.5098
  bertscore: 0.7382


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Helsinki-NLP:
  bleu: 36.2362
  chrf: 64.2543
  meteor: 0.6668
  bertscore: 0.8966


## 5. Qualitative Examples

In [8]:
for i in range(3):
    print(f'\n{"="*60}\nExample {i+1}\n{"="*60}')
    print(f'Source (EN):    {test_en[i]}')
    print(f'Reference (DE): {test_de[i]}')
    print(f'Seq2Seq:        {s2s_trans[i]}')
    print(f'Helsinki-NLP:   {hel_trans[i]}')


Example 1
Source (EN):    A man in an orange hat starring at something.
Reference (DE): Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Seq2Seq:        ein mann mit einem orangefarbenen hut starrt etwas etwas.
Helsinki-NLP:   Ein Mann in einem orangenen Hut, der mit etwas zu tun hat.

Example 2
Source (EN):    A Boston Terrier is running on lush green grass in front of a white fence.
Reference (DE): Ein Boston Terrier läuft über saftig-grünes Gras vor einem weißen Zaun.
Seq2Seq:        ein <unk> läuft auf einem grünen gras vor einem grünen gras. gras.
Helsinki-NLP:   Ein Boston Terrier läuft auf üppig grünem Gras vor einem weißen Zaun.

Example 3
Source (EN):    A girl in karate uniform breaking a stick with a front kick.
Reference (DE): Ein Mädchen in einem Karateanzug bricht ein Brett mit einem Tritt.
Seq2Seq:        ein mädchen in karateanzügen uniform einen einen mit einem <unk>
Helsinki-NLP:   Ein Mädchen in Karate-Uniform bricht einen Stock mit einem Front-Kick.


## 6. Final Results Summary (for LaTeX Report)

In [9]:
print('\n' + '='*60)
print('  TASK 4 — FINAL RESULTS (Copy to LaTeX)')
print('='*60)
print(f'{"Model":<20} {"BLEU":<10} {"METEOR":<10} {"ChrF":<10} {"BERTScore":<10}')
print('-'*60)
print(f'{"Seq2Seq+Attention":<20} {s2s_res["bleu"]:<10.2f} {s2s_res["meteor"]:<10.4f} {s2s_res["chrf"]:<10.2f} {s2s_res["bertscore"]:<10.4f}')
print(f'{"Helsinki-NLP":<20} {hel_res["bleu"]:<10.2f} {hel_res["meteor"]:<10.4f} {hel_res["chrf"]:<10.2f} {hel_res["bertscore"]:<10.4f}')
print('-'*60)


  TASK 4 — FINAL RESULTS (Copy to LaTeX)
Model                BLEU       METEOR     ChrF       BERTScore 
------------------------------------------------------------
Seq2Seq+Attention    3.75       0.5098     39.84      0.7382    
Helsinki-NLP         36.24      0.6668     64.25      0.8966    
------------------------------------------------------------
